<a href="https://colab.research.google.com/github/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`seas-square-ocean-named after.csv`](https://github.com/Prashkov1ch/python-ai-Prashkovich-Anna/blob/main/data/seas-square-ocean-named%20after.csv) — информация о морях:
  - `sea` — Wikidata ID сущности
  - `seaLabel` — название моря/океана на русском
  - `oceanLabel` — принадлежность к океану
  - `area` — площадь (км²)
  - `depth` — глубина (м)
  - `coordinates` — гео-координаты в формате WKT
  - `named_afterLabel` — объект, в честь которого названо

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем CSV-файл с данными об океанах в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных, типы и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [20]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Prashkovich-Anna"  # 🔧 ИЗМЕНЕНО: ваше имя репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Prashkovich-Anna
    !git clone -q https://github.com/Prashkov1ch/python-ai-Prashkovich-Anna.git  # 🔧 ИЗМЕНЕНО: ваш URL

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Prashkovich-Anna


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [21]:
# 🐱 Шаг 2A. Чтение CSV-файла в pandas

import pandas as pd

# 🔧 ИЗМЕНЕНО: читаем ваш файл с данными об океанах
# Обратите внимание: в имени файла есть пробел — pandas справится, но будьте внимательны
df_oceans = pd.read_csv("data/seas-square-ocean-named after.csv")

print("✅ Загружено строк в df_oceans:", len(df_oceans))
print("✅ Колонки:", df_oceans.columns.tolist())

✅ Загружено строк в df_oceans: 214
✅ Колонки: ['sea', 'seaLabel', 'oceanLabel', 'area', 'depth', 'coordinates', 'named_afterLabel']


## 🧹 [2B] Очистка, переименование + Подготовка выборок для анализа

### 🔧 Что мы делаем в этом шаге:

1. **Переименовываем технические столбцы:**
   - `sea` → `URL` (сохраняем для верификации!)
   - `seaLabel` → `sea`, `oceanLabel` → `ocean`, `named_afterLabel` → `named_after`

2. **Преобразуем числовые столбцы:**
   - `area`, `depth` → числовой тип через `pd.to_numeric(errors="coerce")`
   - **Важно:** Не используем `fillna(0)` — NaN остаётся «нет данных», а не «0»

3. **Проверяем уникальность морей:**
   - Сравниваем `len(df)` и `df["URL"].nunique()`
   - Создаём `df_unique = df.drop_duplicates(subset="URL")`

4. **🆕 Готовим аналитические выборки для будущих графиков:**

| Выборка | Фильтр | Зачем нужна |
|---------|--------|-------------|
| `df_area` | `area.notna()` | Графики по площади (ось X: «n=X морей») |
| `df_depth` | `depth.notna()` | Графики по глубине (ось Y: «n=X морей») |
| `df_named` | `named_after.notna()` | Анализ источников названий |

> ⚠️ **Важно:** Все последующие графики и статистика должны использовать эти выборки, а не полный `df_unique`. Это гарантирует честные подписи: «График построен по 214 морям с известной глубиной», а не «по 214 морям (из них 50 с пропущенной глубиной)».

**Итоговые объекты в памяти:**
- `df_oceans` — исходные данные после очистки (не использовать для анализа!)
- `df_unique` — данные без дубликатов по URL
- `df_area`, `df_depth`, `df_named` — готовые выборки для конкретных задач

In [22]:
# 🧹 Шаг 2B. Очистка, переименование + Подготовка выборок для графиков

import pandas as pd
import os

# 🔧 ПРОВЕРКА: существует ли df_oceans? Если нет — загружаем автоматически
try:
    df_oceans
except NameError:
    print("⚠️ df_oceans не найден! Автоматически загружаем из CSV...")
    csv_path = "data/seas-square-ocean-named after.csv"
    if os.path.exists(csv_path):
        df_oceans = pd.read_csv(csv_path)
        print(f"✅ Загружено: {len(df_oceans)} строк")
    else:
        raise FileNotFoundError("❌ CSV-файл не найден! Сначала запустите ячейку [2A]")

print("📋 Исходные колонки:", df_oceans.columns.tolist())
print()

# 1) Переименовываем технический столбец с URL Wikidata (НЕ удаляем!)
if "sea" in df_oceans.columns:
    df_oceans = df_oceans.rename(columns={"sea": "URL"})
    print("✅ Столбец 'sea' переименован в 'URL' (сохранён для верификации)")
else:
    print("⏭️ Столбец 'sea' не найден — пропускаем переименование")

# 2) Переименовываем столбцы: убираем суффикс Label
label_columns = {
    "seaLabel": "sea",
    "oceanLabel": "ocean",
    "named_afterLabel": "named_after"
}
columns_to_rename = {k: v for k, v in label_columns.items() if k in df_oceans.columns}

if columns_to_rename:
    df_oceans = df_oceans.rename(columns=columns_to_rename)
    print(f"✅ Столбцы переименованы: {list(columns_to_rename.keys())} → без суффикса")
else:
    print("⏭️ Столбцы *Label не найдены — пропускаем переименование")

# 3) Приводим числовые столбцы к числовому типу (БЕЗ fillna(0)!)
if "area" in df_oceans.columns:
    df_oceans["area"] = pd.to_numeric(df_oceans["area"], errors="coerce")
    print("✅ Столбец 'area' преобразован в числовой (NaN сохранён)")

if "depth" in df_oceans.columns:
    df_oceans["depth"] = pd.to_numeric(df_oceans["depth"], errors="coerce")
    print("✅ Столбец 'depth' преобразован в числовой (NaN сохранён)")

# 4) 🔍 ПРОВЕРКА УНИКАЛЬНОСТИ URL
print("\n" + "="*60)
print("🔍 ПРОВЕРКА УНИКАЛЬНОСТИ МОРЕЙ")
print("="*60)

total_rows = len(df_oceans)
unique_urls = df_oceans["URL"].nunique()

print(f"Всего строк: {total_rows}")
print(f"Уникальных морей (по URL): {unique_urls}")

if total_rows == unique_urls:
    print("✅ Отлично: одна строка на каждое море (дубликатов нет)")
else:
    print(f"⚠️ Найдено дубликатов: {total_rows - unique_urls} строк")

# 5) Создаём df_unique для всей дальнейшей статистики
df_unique = df_oceans.drop_duplicates(subset="URL")
print(f"\n✅ Создан df_unique: {len(df_unique)} строк")

# 6) 🆕 ПОДГОТОВКА АНАЛИТИЧЕСКИХ ВЫБОРОК ДЛЯ БУДУЩИХ ГРАФИКОВ
print("\n" + "="*60)
print("📊 ПОДГОТОВКА ВЫБОРОК ДЛЯ ГРАФИКОВ (Задание 3)")
print("="*60)

# Выборка для графиков по площади
df_area = df_unique[df_unique["area"].notna()].copy()
print(f"📏 Морей с известной площадью: {len(df_area)}")

# Выборка для графиков по глубине
df_depth = df_unique[df_unique["depth"].notna()].copy()
print(f"📏 Морей с известной глубиной: {len(df_depth)}")

# Выборка для анализа источников названий
df_named = df_unique[df_unique["named_after"].notna()].copy()
print(f"📏 Морей с данными об именовании: {len(df_named)}")

# 7) Статистика пропусков (по df_unique)
print("\n📊 Статистика пропусков (NaN) в df_unique:")
null_counts = df_unique.isnull().sum()
for col, count in null_counts.items():
    pct = count / len(df_unique) * 100
    if count > 0:
        print(f"   {col}: {count} пропусков ({pct:.1f}%)")
    else:
        print(f"   {col}: ✅ без пропусков")

# 8) 🔗 Примеры URL для проверки
if "URL" in df_unique.columns:
    print("\n🔗 Примеры URL для верификации данных:")
    for url in df_unique["URL"].head(3):
        print(f"   {url}")

print("\n" + "="*60)
print("✅ Данные готовы к анализу!")
print("📋 Используйте:")
print("   • df_unique — для общей статистики")
print("   • df_area   — для графиков по площади")
print("   • df_depth  — для графиков по глубине")
print("   • df_named  — для анализа источников названий")
print("="*60)

📋 Исходные колонки: ['sea', 'seaLabel', 'oceanLabel', 'area', 'depth', 'coordinates', 'named_afterLabel']

✅ Столбец 'sea' переименован в 'URL' (сохранён для верификации)
✅ Столбцы переименованы: ['seaLabel', 'oceanLabel', 'named_afterLabel'] → без суффикса
✅ Столбец 'area' преобразован в числовой (NaN сохранён)
✅ Столбец 'depth' преобразован в числовой (NaN сохранён)

🔍 ПРОВЕРКА УНИКАЛЬНОСТИ МОРЕЙ
Всего строк: 214
Уникальных морей (по URL): 182
⚠️ Найдено дубликатов: 32 строк

✅ Создан df_unique: 182 строк

📊 ПОДГОТОВКА ВЫБОРОК ДЛЯ ГРАФИКОВ (Задание 3)
📏 Морей с известной площадью: 42
📏 Морей с известной глубиной: 49
📏 Морей с данными об именовании: 44

📊 Статистика пропусков (NaN) в df_unique:
   URL: ✅ без пропусков
   sea: ✅ без пропусков
   ocean: 63 пропусков (34.6%)
   area: 140 пропусков (76.9%)
   depth: 133 пропусков (73.1%)
   coordinates: 33 пропусков (18.1%)
   named_after: 138 пропусков (75.8%)

🔗 Примеры URL для верификации данных:
   http://www.wikidata.org/entity/Q1591

## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными об океанах:

- посмотрим размер таблицы (количество строк и столбцов);
- выведем список всех столбцов;
- посмотрим первые несколько строк для понимания структуры;
- посчитаем базовую статистику по числовым полям (`area`, `depth`).

**Зачем это нужно:**
- Убедиться, что данные загрузились корректно.
- Понять, какие столбцы требуют очистки или переименования.
- Оценить диапазоны числовых значений (площадь, глубина).
- Выявить очевидные проблемы (пропуски, странные форматы).

**Что мы увидим:**
| Показатель | Описание |
|------------|----------|
| `shape` | Количество строк и столбцов в таблице |
| `columns` | Список всех имён столбцов |
| `head()` | Первые 5 строк для быстрого просмотра |
| `describe()` | Статистика по числовым колонкам (min, max, mean, std) |

> **Важно:** Этот шаг выполняется **до очистки данных**, чтобы увидеть исходное состояние и потом сравнить с результатом после Шага 2B.

In [23]:
# 🔍 Шаг 3. Обзор данных: структура и первые строки
# 🔧 ИЗМЕНЕНО: используем df_unique вместо df_oceans

def show_info(df, name, n=5):
    """Краткий обзор DataFrame"""
    print(f"\n{'='*60}")
    print(f"📊 {name}")
    print('='*60)
    print("📏 Размер таблицы (строки, столбцы):", df.shape)
    print("📋 Столбцы:", ", ".join(df.columns))
    print(f"\n🔢 Первые {n} строк:")
    display(df.head(n))

    null_counts = df.isnull().sum()
    if null_counts.any():
        print("\n⚠️ Пропуски в столбцах:")
        for col, count in null_counts[null_counts > 0].items():
            print(f"   {col}: {count} пропусков ({count/len(df)*100:.1f}%)")
    else:
        print("\n✅ Пропусков не обнаружено")

# 🚀 Запуск обзора для df_unique (не df_oceans!)
show_info(df_unique, "Моря и океаны (df_unique)", n=5)

# 📈 Бонус: базовая статистика по числовым колонкам (по df_unique!)
print("\n" + "= "*60)
print("📈 Базовая статистика по числовым столбцам")
print("= "*60)

numeric_cols = df_unique.select_dtypes(include=['number']).columns
if len(numeric_cols) > 0:
    print(f"Числовые столбцы: {', '.join(numeric_cols)}\n")
    print(df_unique[numeric_cols].describe().round(2))
else:
    print("⚠️ Числовые столбцы не найдены")


📊 Моря и океаны (df_unique)
📏 Размер таблицы (строки, столбцы): (182, 7)
📋 Столбцы: URL, sea, ocean, area, depth, coordinates, named_after

🔢 Первые 5 строк:


,URL,sea,ocean,area,depth,coordinates,named_after
0,http://www.wikidata.org/entity/Q159183,Филиппинское море,Тихий океан,5000000.0,10911.0,Point(134.0 18.0),NaN
1,http://www.wikidata.org/entity/Q58705,Аравийское море,Индийский океан,3862000.0,5800.0,Point(64.0 16.0),NaN
2,http://www.wikidata.org/entity/Q37660,Южно-Китайское море,Тихий океан,3500000.0,5559.0,Point(113.0 12.0),NaN
4,http://www.wikidata.org/entity/Q1247,Карибское море,Северная Атлантика,2754000.0,7686.0,Point(-75.0 15.0),NaN
6,http://www.wikidata.org/entity/Q33254,Тасманово море,Тихий океан,2300000.0,5994.0,Point(161.0 -37.0),Абел Янсзон Тасман



⚠️ Пропуски в столбцах:
   ocean: 63 пропусков (34.6%)
   area: 140 пропусков (76.9%)
   depth: 133 пропусков (73.1%)
   coordinates: 33 пропусков (18.1%)
   named_after: 138 пропусков (75.8%)

= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
📈 Базовая статистика по числовым столбцам
= = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
Числовые столбцы: area, depth

             area     depth
count       42.00     49.00
mean    882304.00   2722.17
std    1113395.32   2640.49
min        210.00      5.00
25%     242750.00    293.00
50%     457908.00   2100.00
75%     955000.00   4191.00
max    5000000.00  10911.00


## ## 🌊 [4] Уникальный анализ: группировка по океанам и источники названий

**Цель шага**: Найти интересные паттерны в данных, специфичные для морей и океанов.

### 🔍 Что мы анализируем:

#### 1️⃣ Группировка по океану (`ocean`)
Используем `groupby()` + агрегатные функции, чтобы ответить на вопросы:
- В каком океане больше всего морей?
- Какой океан имеет наибольшую **среднюю площадь** моря?
- Какова **суммарная площадь** всех морей в каждом океане?
- Какая **максимальная глубина** зафиксирована в каждом океане?

| Метод | Описание |
|-------|----------|
| `df.groupby("ocean")` | Группирует строки по значению в столбце `ocean` |
| `.size()` | Считает количество строк в каждой группе |
| `.mean()`, `.sum()`, `.max()` | Считают среднее, сумму, максимум по числовым колонкам |
| `.sort_values()` | Сортирует результат по выбранному столбцу |

#### 2️⃣ Топ источников названий (`named_after`)
Анализируем столбец `named_after` (только непустые значения):
- В честь кого или чего чаще всего называют моря?
- Есть ли закономерности: страны, люди, географические направления?

> ⚠️ **Важно**:
> - Используем `df_unique` для всех расчётов (без дубликатов по URL)
> - Для анализа площади используем `df_area` (где `area.notna()`)
> - Для анализа `named_after` используем `df_named` (где `named_after.notna()`)
> - Пропуски (NaN) не заменяем на 0 — они честно означают «данных нет»

### 📋 Формат вывода:
- Все результаты — в виде таблиц (`pandas DataFrame`)
- Текстовые выводы с интерпретацией: «Наибольшее количество морей в ...»
- Готовность к копированию в отчёт или экспорт в CSV

In [25]:
# 🌊 Шаг 4. Уникальный анализ: группировка по океанам + источники названий

import pandas as pd

print("🌊 УНИКАЛЬНЫЙ АНАЛИЗ ДАННЫХ ОБ ОКЕАНАХ")
print("="*70)

# ============================================================================
# ЧАСТЬ 1: Группировка по океану (ocean)
# ============================================================================
print("\n📊 1. АНАЛИЗ ПО ОКЕАНАМ (группировка + статистика)")
print("-"*70)

# Проверяем, есть ли столбец ocean и данные в нём
if "ocean" in df_unique.columns and df_unique["ocean"].notna().any():

    # 🔹 Статистика: сколько морей в каждом океане
    print("📋 Количество морей по океанам:")
    oceans_count = df_unique["ocean"].value_counts().reset_index()
    oceans_count.columns = ["Океан", "Количество морей"]
    oceans_count = oceans_count.sort_values("Количество морей", ascending=False)
    print(oceans_count.to_string(index=False))

    # 🔹 Средняя площадь моря в каждом океане (только где area известна!)
    print("\n📏 Средняя площадь моря по океанам (только моря с известной площадью):")

    # Используем df_area, где area.notna()
    if "area" in df_area.columns and len(df_area) > 0:
        area_by_ocean = df_area.groupby("ocean")["area"].agg([
            ("Количество", "count"),
            ("Средняя площадь, км²", "mean"),
            ("Медиана, км²", "median"),
            ("Мин, км²", "min"),
            ("Макс, км²", "max")
        ]).round(0)

        # Сортируем по средней площади (по убыванию)
        area_by_ocean = area_by_ocean.sort_values("Средняя площадь, км²", ascending=False)
        print(area_by_ocean.to_string())
    else:
        print("⚠️ Нет данных о площади для расчёта")

    # 🔹 Суммарная площадь всех морей в каждом океане
    print("\n🗂️ Суммарная площадь морей по океанам:")
    if "area" in df_area.columns and len(df_area) > 0:
        total_area = df_area.groupby("ocean")["area"].sum().reset_index()
        total_area.columns = ["Океан", "Суммарная площадь, км²"]
        total_area["Суммарная площадь, км²"] = total_area["Суммарная площадь, км²"].round(0).astype(int)
        total_area = total_area.sort_values("Суммарная площадь, км²", ascending=False)
        print(total_area.to_string(index=False))

    # 🔹 Максимальная глубина по океанам
    print("\n📉 Максимальная глубина по океанам (только моря с известной глубиной):")
    if "depth" in df_depth.columns and len(df_depth) > 0:
        max_depth = df_depth.groupby("ocean")["depth"].max().reset_index()
        max_depth.columns = ["Океан", "Макс. глубина, м"]
        max_depth = max_depth.sort_values("Макс. глубина, м", ascending=False)
        print(max_depth.to_string(index=False))

    # 🔹 Текстовые выводы-инсайты
    print("\n💡 Ключевые инсайты по океанам:")

    # Какой океан имеет больше всего морей?
    top_ocean_by_count = oceans_count.iloc[0]
    print(f"   • Больше всего морей в океане «{top_ocean_by_count['Океан']}»: {top_ocean_by_count['Количество морей']} морей")

    # Какой океан имеет наибольшую среднюю площадь моря?
    if "area" in df_area.columns and len(df_area) > 0:
        top_ocean_by_area = area_by_ocean.iloc[0]
        print(f"   • Наибольшая средняя площадь моря: «{top_ocean_by_area.name}» — {top_ocean_by_area['Средняя площадь, км²']:,.0f} км²")

    # Какой океан имеет самое глубокое море?
    if "depth" in df_depth.columns and len(df_depth) > 0:
        deepest = max_depth.iloc[0]
        print(f"   • Самое глубокое море находится в «{deepest['Океан']}»: {deepest['Макс. глубина, м']:,.0f} м")

else:
    print("⚠️ Столбец 'ocean' пуст или отсутствует — пропускаем группировку")

# ============================================================================
# ЧАСТЬ 2: Анализ источников названий (named_after)
# ============================================================================
print("\n" + "="*70)
print("🔤 2. АНАЛИЗ ИСТОЧНИКОВ НАЗВАНИЙ (named_after)")
print("-"*70)

if "named_after" in df_named.columns and len(df_named) > 0:

    # 🔹 Топ-10 самых частых источников названий
    print("📋 Топ-10 объектов, в честь которых чаще всего называют моря:")
    top_named = df_named["named_after"].value_counts().head(10).reset_index()
    top_named.columns = ["Объект (named_after)", "Количество морей"]
    top_named["Ранг"] = range(1, len(top_named)+1)
    top_named = top_named[["Ранг", "Объект (named_after)", "Количество морей"]]
    print(top_named.to_string(index=False))

    # 🔹 Статистика по уникальности
    print(f"\n📊 Статистика:")
    print(f"   • Всего морей с известным источником названия: {len(df_named)}")
    print(f"   • Уникальных источников названий: {df_named['named_after'].nunique()}")
    print(f"   • Доля морей с известным named_after: {len(df_named)/len(df_unique)*100:.1f}% от всех морей")

    # 🔹 Простая категоризация (эвристики)
    print("\n🔍 Быстрая категоризация источников названий:")

    def simple_category(name):
        name_lower = str(name).lower()
        # Направления
        if any(d in name_lower for d in ["север", "юг", "запад", "восток"]):
            return "🧭 Направление"
        # Цвета
        if any(c in name_lower for c in ["красн", "бел", "чёрн", "жёлт", "син"]):
            return "🎨 Цвет"
        # Люди (суффиксы)
        if any(s in name_lower for s in ["ов ", "ев ", "ин ", "вич", "на ", "ский"]):
            return "👤 Человек"
        # Страны/регионы
        if any(r in name_lower for r in ["индия", "китай", "япония", "европ", "арктик", "атлант"]):
            return "🌍 Регион"
        return "❓ Другое"

    df_named_temp = df_named.copy()
    df_named_temp["category"] = df_named_temp["named_after"].apply(simple_category)

    category_stats = df_named_temp["category"].value_counts().reset_index()
    category_stats.columns = ["Категория", "Количество"]
    category_stats["Процент"] = (category_stats["Количество"] / len(df_named) * 100).round(1).astype(str) + "%"
    print(category_stats.to_string(index=False))

    # 🔹 Текстовый инсайт
    top_cat = category_stats.iloc[0]
    print(f"\n💡 Инсайт: Чаще всего моря называют по «{top_cat['Категория']}» ({top_cat['Процент']})")

    # 🔹 Бонус: какие океаны чаще получают названия по направлениям?
    if "🧭 Направление" in df_named_temp["category"].values and "ocean" in df_named_temp.columns:
        print("\n🧭 Моря, названные по направлениям, по океанам:")
        direction_seas = df_named_temp[df_named_temp["category"] == "🧭 Направление"]
        if "ocean" in direction_seas.columns:
            dir_by_ocean = direction_seas["ocean"].value_counts().head(5)
            for ocean, count in dir_by_ocean.items():
                print(f"   • {ocean}: {count} морей")

else:
    print("⚠️ Нет данных в столбце 'named_after' для анализа")

# ============================================================================
# ФИНАЛ: Сводка
# ============================================================================
print("\n" + "="*70)
print("✅ Уникальный анализ завершён!")
print("📋 Использованные выборки:")
print(f"   • df_unique: {len(df_unique)} морей (общая статистика)")
print(f"   • df_area: {len(df_area)} морей с известной площадью")
print(f"   • df_depth: {len(df_depth)} морей с известной глубиной")
print(f"   • df_named: {len(df_named)} морей с известным источником названия")
print("="*70)

🌊 УНИКАЛЬНЫЙ АНАЛИЗ ДАННЫХ ОБ ОКЕАНАХ

📊 1. АНАЛИЗ ПО ОКЕАНАМ (группировка + статистика)
----------------------------------------------------------------------
📋 Количество морей по океанам:
                    Океан  Количество морей
              Тихий океан                21
          Индийский океан                14
 Северный Ледовитый океан                12
       Северная Атлантика                10
         Средиземное море                 9
              Южный океан                 8
      Атлантический океан                 5
            Эгейское море                 4
        Восточная Балтика                 3
Восточное Средиземноморье                 3
            Ваттовое море                 3
      Южно-Китайское море                 3
           Баренцево море                 2
        Левантийское море                 2
    Western Mediterranean                 2
          Балтийское море                 2
       Адриатическое море                 1
              Чёр

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨